In [31]:
import urllib
import time
from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd

DRIVER_PATH = 'chromedriver.exe'
BASE_URL = 'https://map.naver.com/v5/search/'

def collectFamousStore(station):
    
    # 키워드 생성 후 URL로 변환
    keyword = station + "맛집"
    
    encText = urllib.parse.quote(keyword)
    url = BASE_URL + encText
    
    # selenium 브라우저로 해당 URL 접속
    browser = webdriver.Chrome(DRIVER_PATH)
    time.sleep(5)
    browser.get(url)

    # 필요한 정보가 있는 iframe 페이지에 id명을 통한 접근
    browser.switch_to.frame('searchIframe')
    time.sleep(5)

    # 데이터를 담을 리스트와 각 항목에 대한 접근을 확인하기 위한 cnt 변수 생성
    outList = []
    cnt = 0
    
    while True:
        
        # 스크롤을 하면 추가되는 <li> 항목을 모두 가져오기 위한 div 스크롤 조작  
        container = browser.find_element_by_id('_pcmap_list_scroll_container')
        browser.execute_script("arguments[0].scrollBy(0, 3000)", container)                                
        time.sleep(5)
        browser.execute_script("arguments[0].scrollBy(0, 4500)", container)
        time.sleep(5)
        browser.execute_script("arguments[0].scrollBy(0, 5000)", container)
        time.sleep(5)
        
        targets = browser.find_elements_by_css_selector("li[class='_22p-O _2NEjP']")
        
        # 페이지당 50개인 <li> 항목을 모두 가져왔는지 확인 
        print(len(targets))

        for t in targets:
            # 접근 중인 항목 번호 
            cnt += 1
            print(cnt)
            
            pageDetail = t.find_element_by_css_selector('a._3LMxZ')
            pageDetail.click()
            time.sleep(5)
            browser.switch_to.default_content()
            browser.switch_to.frame('entryIframe')
            
            # 스크롤 된 페이지 소스를 bs4 객체로 변환
            htmlStr = browser.page_source
            bs = BeautifulSoup(htmlStr,'html.parser')

            browser.switch_to.default_content()
            browser.switch_to.frame('searchIframe')
        
            targetDiv = bs.select_one("div[role='main']")
            
            # 썸네일 이미지
            img = targetDiv.select_one('#ibu_1')
            if img:
                img = img['style'].split('"')[1]
            else:
                img = None
          
            # 점포명, 분류
            titleInfo = targetDiv.select_one('#_title')
            name = titleInfo.select_one('span._3XamX').text
            category = titleInfo.select_one('span._3ocDE').text
            
            # 별점 (별점이 없으면 None을 저장)
            score = targetDiv.select_one("span[class='_1Y6hi _1A8_M'] em")
            if score:
                score = float(score.text)
            else:
                score = None
            
            reviewInfo = targetDiv.select("span[class='_1Y6hi']")
            visitNum = None
            blogNum = None
            for i in reviewInfo:
                info = i.text.split(' ')
                if info[0] in ['방문자리뷰','주문자리뷰']:
                    visitNum = int(info[1].replace(',',''))
                elif '블로그리뷰' == info[0]:
                    blogNum = int(info[1].replace(',',''))
            
            
            phone = targetDiv.select_one('span._3ZA0S')
            if phone:
                phone = phone.text
            else:
                phone = None

            loc = targetDiv.select_one('span._2yqUQ')
            if loc:
                loc = loc.text
            else:
                loc = None
            
            distance = targetDiv.select_one('div._2P6sT')
            if distance:
                distance = distance.text[1:]
            else:
                distance = None
                
            outList.append([category, name, score, img, visitNum, blogNum, phone, loc, distance, station])

            
        # 탐색이 끝난 후 다음 페이지로 이동
        
        # 이전 페이지, 5개의 페이지 번호, 다음 페이지 버튼으로 구성된
        # 7개의 <a> 태그 중 다음 페이지 버튼인 마지막 <a> 태그 선택
        nextBtn = browser.find_elements_by_css_selector('div._2ky45 a')[-1]
        
        # 해당 버튼의 class명 확인
        isLastPage = nextBtn.get_attribute("class")
        print(isLastPage)

        # 클래스명이 "_2bgjK "이면 (공백 포함 주의 !!)
        # 마지막 페이지가 아니므로 다음 페이지로 이동
        if isLastPage == "_2bgjK ":
            print("this is not last page.")
            nextBtn.click()
            time.sleep(7)
        
        # class명이 "_2bgjK _34lTS"이면 마지막 페이지이므로 반복문 종료
        else:
            break

    # 엑셀에서 한글이 깨지지 않는 'utf-8-sig' 인코딩으로 csv 파일 추출 
    df = pd.DataFrame(outList,columns=['분류','점포명','별점','이미지','방문자 리뷰 수', '블로그 리뷰 수', '전화번호','위치', '역거리','역명'])
    df.to_csv(f"{keyword}CSV.csv",encoding='utf-8-sig')

if __name__ == "__main__":
    collectFamousStore(input())
    
#     stationList = ['범내골역','범일역','좌천역','부산진역','초량역','부산역','중앙역','남포역','자갈치역']
    
#     for station in stationList:
#         # 지하철역명 입력 후 collectFamousStore 함수로 전달 
#         collectFamousStore(station)

범내골역


C:\Users\pnu\AppData\Local\Temp\ipykernel_1320\3822232793.py:19: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  browser = webdriver.Chrome(DRIVER_PATH)
C:\Users\pnu\AppData\Local\Temp\ipykernel_1320\3822232793.py:34: DeprecationWarning: find_element_by_* commands are deprecated. Please use find_element() instead
  container = browser.find_element_by_id('_pcmap_list_scroll_container')
C:\Users\pnu\AppData\Local\Temp\ipykernel_1320\3822232793.py:42: DeprecationWarning: find_elements_by_css_selector is deprecated. Please use find_elements(by=By.CSS_SELECTOR, value=css_selector) instead
  targets = browser.find_elements_by_css_selector("li[class='_22p-O _2NEjP']")
C:\Users\pnu\AppData\Local\Temp\ipykernel_1320\3822232793.py:52: DeprecationWarning: find_element_by_css_selector is deprecated. Please use find_element(by=By.CSS_SELECTOR, value=css_selector) instead
  pageDetail = t.find_element_by_css_selector('a._3LMxZ')


50
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50


C:\Users\pnu\AppData\Local\Temp\ipykernel_1320\3822232793.py:122: DeprecationWarning: find_elements_by_css_selector is deprecated. Please use find_elements(by=By.CSS_SELECTOR, value=css_selector) instead
  nextBtn = browser.find_elements_by_css_selector('div._2ky45 a')[-1]


_2bgjK 
this is not last page.
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
_2bgjK 
this is not last page.
50
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
_2bgjK 
this is not last page.
50
151
152
153
154
155
156
157
158
159
160
161
162


WebDriverException: Message: chrome not reachable
  (Session info: chrome=102.0.5005.115)
Stacktrace:
Backtrace:
	Ordinal0 [0x00D4D953+2414931]
	Ordinal0 [0x00CDF5E1+1963489]
	Ordinal0 [0x00BCC580+836992]
	Ordinal0 [0x00BC01E2+786914]
	Ordinal0 [0x00BC0A18+789016]
	Ordinal0 [0x00BC22F2+795378]
	Ordinal0 [0x00BBBCA9+769193]
	Ordinal0 [0x00BCDA30+842288]
	Ordinal0 [0x00C23FF2+1196018]
	Ordinal0 [0x00C142B6+1131190]
	Ordinal0 [0x00BEE860+976992]
	Ordinal0 [0x00BEF756+980822]
	GetHandleVerifier [0x00FBCC62+2510274]
	GetHandleVerifier [0x00FAF760+2455744]
	GetHandleVerifier [0x00DDEABA+551962]
	GetHandleVerifier [0x00DDD916+547446]
	Ordinal0 [0x00CE5F3B+1990459]
	Ordinal0 [0x00CEA898+2009240]
	Ordinal0 [0x00CEA985+2009477]
	Ordinal0 [0x00CF3AD1+2046673]
	BaseThreadInitThunk [0x7667FA29+25]
	RtlGetAppContainerNamedObjectPath [0x77147A7E+286]
	RtlGetAppContainerNamedObjectPath [0x77147A4E+238]
